In [1]:
from pyspark.sql.functions import (
    col, trim, upper, when, current_timestamp,
    lit, to_date, datediff, round as spark_round
)
from pyspark.sql.types import DecimalType, IntegerType
from datetime import datetime

BRONZE_TABLE  = "Interac_Bronze.dbo.fraud_labels"
SILVER_TABLE  = "silver_fraud_labels"
SILVER_DB     = "Interac_Fabric_Workspace.Interac_Silver.dbo"
PIPELINE_NAME = "NB_07_Silver_FraudLabels"
BATCH_DATE    = datetime.now().strftime("%Y-%m-%d")

print(f"Silver Fraud Labels Pipeline")
print(f"Started: {datetime.now()}")

StatementMeta(, 606123ba-a03e-4f76-bd1e-e946260d4913, 3, Finished, Available, Finished, False)

Silver Fraud Labels Pipeline
Started: 2026-05-06 00:28:22.542518


In [2]:
df_bronze = spark.read.table(BRONZE_TABLE)
total_bronze = df_bronze.count()
print(f"Bronze rows read: {total_bronze:,}")

dq_results = {}
dq_results["null_fraud_case_id"] = df_bronze.filter(col("fraud_case_id").isNull()).count()
dq_results["null_cardholder_id"] = df_bronze.filter(col("cardholder_id").isNull()).count()
dq_results["null_amount_at_risk"] = df_bronze.filter(col("amount_at_risk_cad").isNull()).count()
dq_results["confirmed_fraud"] = df_bronze.filter(
    col("confirmation_status") == "CONFIRMED_FRAUD").count()
dq_results["false_positives"] = df_bronze.filter(
    col("confirmation_status") == "FALSE_POSITIVE").count()
dq_results["under_investigation"] = df_bronze.filter(
    col("confirmation_status") == "UNDER_INVESTIGATION").count()
dq_results["critical_severity"] = df_bronze.filter(col("severity") == "CRITICAL").count()
dq_results["not_recovered"] = df_bronze.filter(col("is_recovered") == "N").count()
dq_results["ml_score_below_threshold"] = df_bronze.filter(
    col("ml_fraud_score").cast(DecimalType(10,6)) < 0.5).count()

print("\nDQ CHECK RESULTS:")
print("-" * 45)
for check, count_val in dq_results.items():
    status = "⚠ FLAGGED" if count_val > 0 else "✓ PASSED"
    print(f"{check:<35} {count_val:>6,}  {status}")

StatementMeta(, 606123ba-a03e-4f76-bd1e-e946260d4913, 4, Finished, Available, Finished, False)

Bronze rows read: 2,800

DQ CHECK RESULTS:
---------------------------------------------
null_fraud_case_id                       0  ✓ PASSED
null_cardholder_id                       0  ✓ PASSED
null_amount_at_risk                      0  ✓ PASSED
confirmed_fraud                      1,603  ⚠ FLAGGED
false_positives                        629  ⚠ FLAGGED
under_investigation                    394  ⚠ FLAGGED
critical_severity                      289  ⚠ FLAGGED
not_recovered                        1,810  ⚠ FLAGGED
ml_score_below_threshold                 0  ✓ PASSED


In [3]:
df_quarantine = df_bronze.filter(
    col("fraud_case_id").isNull() |
    col("cardholder_id").isNull() |
    col("amount_at_risk_cad").isNull()
)
quarantine_count = df_quarantine.count()

if quarantine_count > 0:
    (df_quarantine
        .withColumn("_quarantine_reason", lit("NULL_PRIMARY_KEY_OR_AMOUNT"))
        .withColumn("_quarantined_at", current_timestamp())
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_DB}.silver_fraud_labels_quarantine"))
    print(f"Quarantined: {quarantine_count:,} records")

df_valid = df_bronze.filter(
    col("fraud_case_id").isNotNull() &
    col("cardholder_id").isNotNull() &
    col("amount_at_risk_cad").isNotNull()
)

df_silver = (df_valid
    .withColumn("fraud_case_id",       trim(col("fraud_case_id")))
    .withColumn("cardholder_id",       trim(col("cardholder_id")))
    .withColumn("merchant_id",         trim(col("merchant_id")))
    .withColumn("fraud_type",          upper(trim(col("fraud_type"))))
    .withColumn("detection_method",    upper(trim(col("detection_method"))))
    .withColumn("rule_triggered",      upper(trim(col("rule_triggered"))))
    .withColumn("confirmation_status", upper(trim(col("confirmation_status"))))
    .withColumn("severity",            upper(trim(col("severity"))))
    .withColumn("is_recovered",        upper(trim(col("is_recovered"))))
    .withColumn("reported_to_police",  upper(trim(col("reported_to_police"))))
    .withColumn("case_closed",         upper(trim(col("case_closed"))))
    .withColumn("currency",            upper(trim(col("currency"))))
    .withColumn("transaction_date",
        to_date(col("transaction_date"), "yyyy-MM-dd"))
    .withColumn("detection_date",
        to_date(col("detection_date"), "yyyy-MM-dd"))
    .withColumn("amount_at_risk_cad",
        col("amount_at_risk_cad").cast(DecimalType(18, 2)))
    .withColumn("amount_lost_cad",
        col("amount_lost_cad").cast(DecimalType(18, 2)))
    .withColumn("recovered_amount_cad",
        col("recovered_amount_cad").cast(DecimalType(18, 2)))
    .withColumn("ml_fraud_score",
        col("ml_fraud_score").cast(DecimalType(10, 6)))
    .withColumn("days_to_detect",
        datediff(col("detection_date"), col("transaction_date")))
    .withColumn("net_loss_cad",
        spark_round(col("amount_lost_cad") - col("recovered_amount_cad"), 2))
    .withColumn("recovery_rate_pct",
        when(col("amount_lost_cad") > 0,
            spark_round(
                col("recovered_amount_cad") /
                col("amount_lost_cad") * 100, 2))
        .otherwise(None))
    .withColumn("is_confirmed_fraud",
        when(col("confirmation_status") == "CONFIRMED_FRAUD", "Y").otherwise("N"))
    .withColumn("is_false_positive",
        when(col("confirmation_status") == "FALSE_POSITIVE", "Y").otherwise("N"))
    .withColumn("ml_risk_band",
        when(col("ml_fraud_score") >= 0.9, "VERY_HIGH")
        .when(col("ml_fraud_score") >= 0.7, "HIGH")
        .when(col("ml_fraud_score") >= 0.5, "MEDIUM")
        .otherwise("LOW"))
    .withColumn("_silver_loaded_at", current_timestamp())
    .withColumn("_pipeline_name",    lit(PIPELINE_NAME))
    .withColumn("_batch_date",       lit(BATCH_DATE))
    .drop("_ingested_at", "_source_file", "_lakehouse")
)

print(f"Valid records: {df_silver.count():,}")

StatementMeta(, 606123ba-a03e-4f76-bd1e-e946260d4913, 5, Finished, Available, Finished, False)

Valid records: 2,800


In [4]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .saveAsTable(f"{SILVER_DB}.{SILVER_TABLE}"))

spark.sql(f"OPTIMIZE {SILVER_DB}.{SILVER_TABLE} ZORDER BY (detection_date, cardholder_id)")

final_count = spark.read.table(f"{SILVER_DB}.{SILVER_TABLE}").count()

confirmed = spark.sql(f"""
    SELECT COUNT(*) as c FROM {SILVER_DB}.{SILVER_TABLE}
    WHERE confirmation_status = 'CONFIRMED_FRAUD'
""").collect()[0]["c"]

false_pos = spark.sql(f"""
    SELECT COUNT(*) as c FROM {SILVER_DB}.{SILVER_TABLE}
    WHERE confirmation_status = 'FALSE_POSITIVE'
""").collect()[0]["c"]

print("\n" + "="*60)
print("SILVER FRAUD LABELS SUMMARY")
print("="*60)
print(f"Bronze rows in      : {total_bronze:,}")
print(f"Quarantined         : {quarantine_count:,}")
print(f"Silver rows out     : {final_count:,}")
print(f"Pass rate           : {round(final_count/total_bronze*100, 2)}%")
print(f"Confirmed fraud     : {confirmed:,}")
print(f"False positives     : {false_pos:,}")
print(f"Table               : {SILVER_DB}.{SILVER_TABLE}")
print(f"Completed at        : {datetime.now()}")
print("="*60)

StatementMeta(, 606123ba-a03e-4f76-bd1e-e946260d4913, 6, Finished, Available, Finished, True)


SILVER FRAUD LABELS SUMMARY
Bronze rows in      : 2,800
Quarantined         : 0
Silver rows out     : 2,800
Pass rate           : 100.0%
Confirmed fraud     : 1,603
False positives     : 629
Table               : Interac_Fabric_Workspace.Interac_Silver.dbo.silver_fraud_labels
Completed at        : 2026-05-06 00:29:18.405654
